# Mamba-JEPA — canonical training

final all-position contrastive objective. Board size is selected once below and defaults to **12×12**.

The final JEPA uses full official-Mamba context/target encoders, an independently constructed EMA target initialized from the context encoder, a linear predictor, hard-disjoint one-token targets, and position-stratified InfoNCE.

The default smoke profile writes to a separate directory and cannot overwrite the full run.

In [ ]:
# Only these controls normally change.
BOARD_SIZE = 12                 # 8, 12, or 16
RUN_PROFILE = "smoke"           # "smoke" or "full"
SEED = 42
RESUME_CHECKPOINT = None        # synced latest.pt, or None
CONFIRM_FULL_RUN = ""           # set to "TRAIN" for a full run

assert BOARD_SIZE in {8, 12, 16}
assert RUN_PROFILE in {"smoke", "full"}
if RUN_PROFILE == "full":
    assert CONFIRM_FULL_RUN == "TRAIN", "Set CONFIRM_FULL_RUN = 'TRAIN'."

## Environment

Mount the Drive backup, install this final repository, and keep all large artifacts in `Master_Thesis_Artifacts`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code_Final"
)
assert (PROJECT_ROOT / "pyproject.toml").is_file(), PROJECT_ROOT
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)],
    check=True,
)
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)

## Locked official Mamba backend

The thesis Mamba runs use the official CUDA backend. The pure-Torch backend remains test-only.

In [ ]:
import importlib.metadata
import subprocess
import sys
import torch

assert torch.cuda.is_available(), "Select a GPU runtime in Colab."
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "packaging", "ninja",
        "causal-conv1d==1.6.2.post1",
        "mamba-ssm==2.3.2.post1",
        "--no-build-isolation",
    ],
    check=True,
)
assert importlib.metadata.version("mamba-ssm") == "2.3.2.post1"
assert importlib.metadata.version("causal-conv1d") == "1.6.2.post1"
print("GPU:", torch.cuda.get_device_name(0))
print("Official Mamba runtime: PASS")

## Resolve and validate the run

This reads the locked protocol and the primary path in the evaluation registry. No model is constructed here.

In [ ]:
from dataclasses import asdict
from othello_thesis.experiments import (
    build_jepa_train_config,
    resolve_run_layout,
    validate_canonical_corpus,
)

layout = resolve_run_layout(
    "mamba", "jepa", BOARD_SIZE, profile=RUN_PROFILE
)
corpus = validate_canonical_corpus(layout.data_dir, BOARD_SIZE)
cfg = build_jepa_train_config(
    "mamba",
    BOARD_SIZE,
    layout,
    profile=RUN_PROFILE,
    seed=SEED,
    resume=RESUME_CHECKPOINT,
)
print("data:", layout.data_dir)
print("local output:", layout.local_out_dir)
print("Drive output:", layout.drive_out_dir)
print("resolved config:")
asdict(cfg)

## Visible model definition

The factory below is the factory used by training. It is not called in this cell. This preserves the exact global RNG trajectory. For JEPA, the displayed constructor makes the target-copy and predictor order explicit.

In [ ]:
import inspect
import torch
from othello_thesis.objectives.jepa import JEPAConfig, OthelloJEPA

def build_mamba_jepa(objective_config: JEPAConfig) -> torch.nn.Module:
    """Construct the final JEPA model exactly once, after seeding."""
    assert objective_config.encoder_architecture == "mamba"
    assert objective_config.mamba_backend == "mamba_ssm"
    assert objective_config.loss_type == "infonce"
    assert objective_config.view_mode == "hard_disjoint_future"
    assert objective_config.prediction_horizon == 1
    assert objective_config.predictor_type == "linear"
    assert objective_config.use_ema_target is True
    return OthelloJEPA(objective_config)

# This is the exact composition used by the factory. It visibly shows:
# context encoder -> fresh target encoder -> state copy/freeze -> predictor.
# Inspecting source consumes no RNG and constructs no preview model.
print(inspect.getsource(OthelloJEPA.__init__))

## Train

This is the only cell that starts training and constructs a model. `smoke` and `full` use separate output paths. For resume, set `RESUME_CHECKPOINT` before resolving the config.

In [ ]:
from othello_thesis.training.mamba_jepa import run_training

print(
    f"START {RUN_PROFILE}: MAMBA-JEPA "
    f"{BOARD_SIZE}x{BOARD_SIZE}",
    flush=True,
)
run_training(cfg, model_factory=build_mamba_jepa)
print("TRAINING FINISHED", flush=True)